In [ ]:
import os
import json
import random
import warnings
import math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing, GlobalAttention, global_add_pool

from rdkit import Chem
from rdkit.Chem import AllChem

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

# ------------------------------
# ------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------
# ------------------------------
SAVE_DIR_BASE = "analysis/multilabel_aug_compare_unified"
os.makedirs(SAVE_DIR_BASE, exist_ok=True)

SDF_PATH = "data/ChemTastesDB/ChemTastesDB_multilabel_properties.sdf"


In [ ]:
# ================================================================
# ================================================================
LABEL_MAP_BIN = {"bitterness": 0, "sweetness": 1}                      # 2-class
LABEL_MAP_TRI = {"bitterness": 0, "sweetness": 1, "tastelessness": 2}   # 3-class

RUN_BINARY = True  # True=binary, False=triclass

def get_current_label_map():
    return LABEL_MAP_BIN if RUN_BINARY else LABEL_MAP_TRI

LABEL_MAP = get_current_label_map()
NUM_CLASSES = len(LABEL_MAP)

def set_mode_binary():
    global RUN_BINARY, LABEL_MAP, NUM_CLASSES
    RUN_BINARY = True
    LABEL_MAP = LABEL_MAP_BIN
    NUM_CLASSES = len(LABEL_MAP)

def set_mode_triclass():
    global RUN_BINARY, LABEL_MAP, NUM_CLASSES
    RUN_BINARY = False
    LABEL_MAP = LABEL_MAP_TRI
    NUM_CLASSES = len(LABEL_MAP)


In [ ]:
# ================================================================
# ================================================================
DO_AUGMENT = False
AUG_MODE = "tasteless_only"

AUG_MAX_EXTRA_PER_MOL = 3
AUG_GAP_RATIO = 1.0
AUG_RMSD_THRESH = 0.35
AUG_NOISE_SIGMA = 0.02
AUG_SEED = 0xF00D

# ================================================================
# ================================================================
BASE_PROP_NAMES = [
    "MolecularWeight", "XLogP", "TPSA", "HBondDonorCount",
    "HBondAcceptorCount", "RotatableBondCount",
    "HeavyAtomCount", "Complexity"
]

PROPERTY_DIM = None
PROP_IMPUTER = None
PROP_SCALER = None
KEEP_COLS = None
FINAL_PROP_NAMES = None


In [ ]:
# ================================================================
# SchNet blocks
# ================================================================
class SchNetInteraction(MessagePassing):
    def __init__(self, hidden_channels, num_filters, num_gaussians, cutoff):
        super().__init__(aggr="add")
        self.hidden_channels = hidden_channels
        self.num_filters = num_filters
        self.num_gaussians = num_gaussians
        self.cutoff = cutoff

        self.filter_network = nn.Sequential(
            nn.Linear(num_gaussians, num_filters),
            nn.Softplus(),
            nn.Linear(num_filters, hidden_channels),
        )
        self.dense = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.Softplus(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        self.register_buffer("offset", torch.linspace(0, cutoff, num_gaussians))
        self.register_buffer("coeff", torch.tensor(-0.5 / (cutoff / num_gaussians) ** 2))

    def forward(self, x, edge_index, edge_attr):
        distances = edge_attr.view(-1, 1)
        gauss = torch.exp(self.coeff * (distances - self.offset) ** 2)
        edge_weight = self.filter_network(gauss)
        return self.propagate(edge_index, x=x, edge_weight=edge_weight)

    def message(self, x_j, edge_weight):
        return x_j * edge_weight

    def update(self, aggr_out, x):
        return x + self.dense(aggr_out)


class SchNet(nn.Module):
    def __init__(
        self,
        hidden_channels=128,
        num_filters=128,
        num_interactions=6,
        num_gaussians=100,
        cutoff=10.0,
        max_z=100,
        use_attention_pool=True,
    ):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.num_interactions = num_interactions
        self.cutoff = cutoff

        self.embedding = nn.Embedding(max_z + 1, hidden_channels)
        self.interactions = nn.ModuleList(
            [
                SchNetInteraction(hidden_channels, num_filters, num_gaussians, cutoff)
                for _ in range(num_interactions)
            ]
        )
        if use_attention_pool:
            gate_nn = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels // 2),
                nn.ReLU(),
                nn.Linear(hidden_channels // 2, 1),
            )
            self.pool = GlobalAttention(gate_nn)
        else:
            self.pool = global_add_pool

    def forward(self, batch):
        x = self.embedding(batch.z)
        for interaction in self.interactions:
            x = interaction(x, batch.edge_index, batch.edge_attr)
        out = self.pool(x, batch.batch) if isinstance(self.pool, GlobalAttention) else self.pool(x, batch.batch)
        return out


# ================================================================
# ================================================================
class PropertyBranch(nn.Module):
    def __init__(self, in_dim, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, out_dim), nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class FusionHeadFiLM(nn.Module):
    """
    FiLM: h3d is conditioned on props (gamma, beta)
    fused = gamma * h3d + beta -> (LayerNorm) -> classifier
    """
    def __init__(self, h3d_dim=128, prop_dim=32, cls_dim=3, use_ln_after=True):
        super().__init__()
        self.to_gamma = nn.Sequential(
            nn.Linear(prop_dim, h3d_dim), nn.ReLU(),
            nn.Linear(h3d_dim, h3d_dim), nn.Tanh()
        )
        self.to_beta = nn.Sequential(
            nn.Linear(prop_dim, h3d_dim), nn.ReLU(),
            nn.Linear(h3d_dim, h3d_dim)
        )
        self.ln_after = nn.LayerNorm(h3d_dim) if use_ln_after else nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(h3d_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, cls_dim)
        )

    def forward(self, h3d, hprop):
        gamma = self.to_gamma(hprop)
        beta = self.to_beta(hprop)
        fused = self.ln_after(gamma * h3d + beta)
        return self.classifier(fused)


class SchNetTasteClassifier_FiLM(nn.Module):
    """
    use_properties=True  -> SchNet + PropertyBranch + FiLM head (3D + Properties)
    use_properties=False -> SchNet + simple head (3D-only)
    """
    def __init__(self, schnet_hidden=128, property_dim=None, num_classes=2, use_properties=True):
        super().__init__()
        self.use_properties = use_properties
        self.schnet = SchNet(hidden_channels=schnet_hidden, use_attention_pool=True)

        if self.use_properties:
            assert property_dim is not None, "PROPERTY_DIM must be set after the SAFE preprocessing fit."
            self.prop = PropertyBranch(property_dim, out_dim=32)
            self.head = FusionHeadFiLM(
                h3d_dim=schnet_hidden, prop_dim=32,
                cls_dim=num_classes, use_ln_after=True
            )
        else:
            self.prop = None
            self.head = nn.Sequential(
                nn.LayerNorm(schnet_hidden),
                nn.Linear(schnet_hidden, 128), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(64, num_classes)
            )

    def forward(self, batch, properties=None):
        h3d = self.schnet(batch)
        if self.use_properties and (properties is not None):
            hp = self.prop(properties)
            return self.head(h3d, hp)
        return self.head(h3d)


In [ ]:
# ================================================================
# Dataset
# ================================================================
class TasteDataset(Dataset):
    """
    Read molecules from the SDF and store graphs (nodes = atoms, edges = distance-based) with labels, properties and group information in self.data.
    """
    def __init__(self, sdf_file, transform=None,
                 label_map=None, edge_cutoff=10.0,
                 try_embed_if_no_conf=True, random_seed=0xF00D):
        self.transform = transform
        self.data = []
        self.edge_cutoff = edge_cutoff
        self.mols = []

        if label_map is None:
            label_map = LABEL_MAP

        norm = {
            "bitter": "bitterness", "bitterness": "bitterness",
            "sweet": "sweetness", "sweetness": "sweetness",
            "tasteless": "tastelessness", "tastelessness": "tastelessness",
            "no taste": "tastelessness", "notaste": "tastelessness",
            "none": "tastelessness", "tastelessness ": "tastelessness",
        }
        self.label_map = {k.lower().strip(): int(v) for k, v in label_map.items()}

        suppl = Chem.SDMolSupplier(sdf_file, sanitize=False, removeHs=False)

        cnt_total = cnt_kept = 0
        cnt_none = cnt_no_label = cnt_not_in_map = cnt_no_conf = 0
        seen_raw_labels = {}

        def normalize_label(raw):
            key = raw.strip().lower()
            key = norm.get(key, key)
            return key

        def get_class_taste_prop(m):
            if m.HasProp("Class taste"):
                return "Class taste"
            for p in m.GetPropNames():
                if p.strip().lower() == "class taste":
                    return p
            return None

        for mol in suppl:
            cnt_total += 1
            if mol is None:
                cnt_none += 1
                continue

            try:
                Chem.SanitizeMol(mol)
            except Exception:
                try:
                    Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_KEKULIZE)
                except Exception:
                    cnt_none += 1
                    continue

            taste_prop = get_class_taste_prop(mol)
            if taste_prop is None:
                cnt_no_label += 1
                continue

            raw = mol.GetProp(taste_prop)
            seen_raw_labels[raw] = seen_raw_labels.get(raw, 0) + 1

            taste_key = normalize_label(raw)
            if taste_key not in self.label_map:
                cnt_not_in_map += 1
                continue
            label = self.label_map[taste_key]

            if mol.GetNumConformers() == 0:
                if not try_embed_if_no_conf:
                    cnt_no_conf += 1
                    continue
                mH = Chem.AddHs(mol, addCoords=True)
                params = AllChem.ETKDGv3()
                params.randomSeed = random_seed
                if AllChem.EmbedMolecule(mH, params) != 0:
                    cnt_no_conf += 1
                    continue
                try:
                    AllChem.UFFOptimizeMolecule(mH)
                except Exception:
                    pass
                mol = mH

            conf = mol.GetConformer()
            pos = conf.GetPositions()
            if pos is None or len(pos) == 0:
                cnt_no_conf += 1
                continue

            atomic_nums = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
            n = len(pos)
            edge_indices, edge_attrs = [], []
            for a in range(n):
                for b in range(a + 1, n):
                    dist = float(np.linalg.norm(pos[a] - pos[b]))
                    if dist < self.edge_cutoff:
                        edge_indices.extend([[a, b], [b, a]])
                        edge_attrs.extend([dist, dist])
            if len(edge_indices) == 0:
                for k in range(n):
                    edge_indices.append([k, k])
                    edge_attrs.append(0.0)

            vals, miss = [], []
            for pn in BASE_PROP_NAMES:
                try:
                    v = float(mol.GetProp(pn))
                    if np.isnan(v):
                        vals.append(np.nan); miss.append(1.0)
                    else:
                        vals.append(v);      miss.append(0.0)
                except Exception:
                    vals.append(np.nan);     miss.append(1.0)

            can_smiles = Chem.MolToSmiles(Chem.RemoveHs(mol))
            self.mols.append(Chem.Mol(mol))
            mol_idx = len(self.mols) - 1

            sample = {
                "pos": torch.tensor(pos, dtype=torch.float32),
                "z": torch.tensor(atomic_nums, dtype=torch.long),
                "edge_index": torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                "edge_attr": torch.tensor(edge_attrs, dtype=torch.float32),
                "properties": torch.tensor(vals, dtype=torch.float32),
                "prop_missing": torch.tensor(miss, dtype=torch.float32),
                "label": torch.tensor([label], dtype=torch.long),
                "group": can_smiles,
                "mol_idx": mol_idx,
            }
            self.data.append(sample)
            cnt_kept += 1

        print(f"[TasteDataset] total={cnt_total}, kept={cnt_kept}, "
              f"none={cnt_none}, no_label={cnt_no_label}, not_in_map={cnt_not_in_map}, no_conf={cnt_no_conf}")
        if seen_raw_labels:
            print("[TasteDataset] Raw label histogram:",
                  {k: v for k, v in sorted(seen_raw_labels.items(), key=lambda x: -x[1])})

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        if self.transform:
            sample = self.transform(sample)
        return sample


def _build_sample_from_mol_and_pos(mol, pos, label, edge_cutoff=10.0):
    n = mol.GetNumAtoms()
    atomic_nums = [atom.GetAtomicNum() for atom in mol.GetAtoms()]

    edge_indices, edge_attrs = [], []
    for a in range(n):
        for b in range(a + 1, n):
            d = float(np.linalg.norm(pos[a] - pos[b]))
            if d < edge_cutoff:
                edge_indices.extend([[a, b], [b, a]])
                edge_attrs.extend([d, d])
    if len(edge_indices) == 0:
        for k in range(n):
            edge_indices.append([k, k])
            edge_attrs.append(0.0)

    vals, miss = [], []
    for name in BASE_PROP_NAMES:
        try:
            v = float(mol.GetProp(name))
            if np.isnan(v):
                vals.append(np.nan); miss.append(1.0)
            else:
                vals.append(v);      miss.append(0.0)
        except Exception:
            vals.append(np.nan);     miss.append(1.0)

    can_smiles = Chem.MolToSmiles(Chem.RemoveHs(mol))
    sample = {
        "pos": torch.tensor(pos, dtype=torch.float32),
        "z": torch.tensor(atomic_nums, dtype=torch.long),
        "edge_index": torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
        "edge_attr": torch.tensor(edge_attrs, dtype=torch.float32),
        "properties": torch.tensor(vals, dtype=torch.float32),
        "prop_missing": torch.tensor(miss, dtype=torch.float32),
        "label": torch.tensor([label], dtype=torch.long),
        "group": can_smiles,
    }
    return sample


In [ ]:
# ================================================================
# ================================================================
def augment_tasteless_only(
    dataset,
    train_idx,
    tasteless_label=2,
    max_extra_per_mol=3,
    gap_ratio=1.0,
    rmsd_thresh=0.35,
    noise_sigma=0.02,
    seed=0xF00D,
    save_aug_sdf_path=None,
    save_aug_csv_path=None,
):
    rng = np.random.default_rng(seed)
    y_train = np.array([dataset.data[i]["label"].item() for i in train_idx])

    classes, counts = np.unique(y_train, return_counts=True)
    count_map = {int(c): int(n) for c, n in zip(classes, counts)}

    if tasteless_label not in count_map:
        print(f"[augment_tasteless_only] tasteless_label={tasteless_label} not in train; no augmentation.")
        return []

    cur_taste = count_map[tasteless_label]
    other_counts = [n for c, n in count_map.items() if c != tasteless_label]
    if len(other_counts) == 0:
        print("[augment_tasteless_only] dataset contains only tasteless; no augmentation needed.")
        return []

    other_max = max(other_counts)
    gap = other_max - cur_taste
    if gap <= 0:
        print(f"[augment_tasteless_only] tasteless is already as large as the other classes (cur={cur_taste}, other_max={other_max}); skipping augmentation.")
        return []

    desired_extra = int(max(1, gap * float(gap_ratio)))

    tasteless_mids = {
        dataset.data[i]["mol_idx"]
        for i in train_idx
        if dataset.data[i]["label"].item() == tasteless_label and ("mol_idx" in dataset.data[i])
    }
    tasteless_mids = list(tasteless_mids)
    if len(tasteless_mids) == 0:
        print("[augment_tasteless_only] no tasteless mol_idx candidates; cannot augment.")
        return []

    max_possible_add = len(tasteless_mids) * max_extra_per_mol
    need = min(desired_extra, max_possible_add)
    if need <= 0:
        print("[augment_tasteless_only] need=0")
        return []

    print(f"[augment_tasteless_only] tasteless: cur={cur_taste}, other_max={other_max}, "
          f"gap={gap}, gap_ratio={gap_ratio}, desired_extra={desired_extra}, "
          f"max_possible={max_possible_add}, actual_need={need}")

    writer = Chem.SDWriter(save_aug_sdf_path) if save_aug_sdf_path is not None else None
    aug_records = []
    added_indices = []
    added_for_taste = 0

    rng.shuffle(tasteless_mids)

    for mid in tasteless_mids:
        if added_for_taste >= need:
            break

        base = Chem.Mol(dataset.mols[mid])
        remain = need - added_for_taste
        quota = int(min(max_extra_per_mol, remain))
        if quota <= 0:
            break

        params = AllChem.ETKDGv3()
        params.randomSeed = int(rng.integers(1, 10**9))
        params.pruneRmsThresh = float(rmsd_thresh)

        conf_ids = AllChem.EmbedMultipleConfs(base, numConfs=max(quota, 1), params=params)
        if not conf_ids:
            continue

        try:
            AllChem.UFFOptimizeMoleculeConfs(base, confIds=list(conf_ids))
        except Exception:
            pass

        for cid in list(conf_ids)[:quota]:
            pos = np.asarray(base.GetConformer(cid).GetPositions(), dtype=float)
            if noise_sigma and noise_sigma > 0:
                pos = pos + rng.normal(0, noise_sigma, size=pos.shape)

            new_sample = _build_sample_from_mol_and_pos(
                base, pos, label=tasteless_label,
                edge_cutoff=getattr(dataset, "edge_cutoff", 10.0),
            )
            new_sample["is_augmented"] = True
            dataset.data.append(new_sample)
            new_idx = len(dataset.data) - 1
            added_indices.append(new_idx)
            added_for_taste += 1

            if writer is not None:
                mol_to_write = Chem.Mol(base)
                conf = mol_to_write.GetConformer(cid)
                for i_atom in range(mol_to_write.GetNumAtoms()):
                    conf.SetAtomPosition(i_atom, pos[i_atom])
                mol_to_write.SetProp("_Name", f"aug_tasteless_mid{mid}_idx{new_idx}")
                mol_to_write.SetProp("augmented_class", str(tasteless_label))
                mol_to_write.SetProp("train_original_mid", str(mid))
                writer.write(mol_to_write)

            if save_aug_csv_path is not None:
                rec = {
                    "new_index": int(new_idx),
                    "class_int": int(tasteless_label),
                    "class_name": "tastelessness",
                    "orig_mid": int(mid),
                    "can_smiles": Chem.MolToSmiles(Chem.RemoveHs(base)),
                }
                for pn in BASE_PROP_NAMES:
                    try:
                        rec[pn] = float(base.GetProp(pn))
                    except Exception:
                        rec[pn] = np.nan
                aug_records.append(rec)

            if added_for_taste >= need:
                break

    print(f"[augment_tasteless_only] tasteless added = {added_for_taste}")

    if writer is not None:
        writer.close()
        print(f"[augment_tasteless_only] SDF written to: {save_aug_sdf_path}")

    if save_aug_csv_path is not None and len(aug_records) > 0:
        df_aug = pd.DataFrame(aug_records)
        df_aug.to_csv(save_aug_csv_path, index=False)
        print(f"[augment_tasteless_only] CSV written to: {save_aug_csv_path}")

    return added_indices


In [ ]:
# ================================================================
# ================================================================
def _fit_property_preprocessors_safe(ds, idx_array, iqr_eps=1e-6):
    global PROP_IMPUTER, PROP_SCALER, KEEP_COLS, PROPERTY_DIM, FINAL_PROP_NAMES

    train_raw = np.stack([ds.data[i]["properties"].numpy() for i in idx_array], axis=0)
    imputer = SimpleImputer(strategy="median").fit(train_raw)
    imp = imputer.transform(train_raw)

    q1 = np.quantile(imp, 0.25, axis=0)
    q3 = np.quantile(imp, 0.75, axis=0)
    iqr = q3 - q1
    KEEP_COLS = np.where(iqr >= iqr_eps)[0]
    if KEEP_COLS.size == 0:
        KEEP_COLS = np.arange(imp.shape[1])

    scaler = RobustScaler().fit(imp[:, KEEP_COLS])

    PROP_IMPUTER = imputer
    PROP_SCALER = scaler
    base_names_short = np.array(["MW", "XLogP", "TPSA", "HBD", "HBA", "RotB", "HeavyAtom", "Complexity"])
    FINAL_PROP_NAMES = base_names_short[KEEP_COLS].tolist()
    PROPERTY_DIM = len(KEEP_COLS)

    print(f"[SAFE PREP] keep_cols={KEEP_COLS.tolist()}  property_dim={PROPERTY_DIM}  names={FINAL_PROP_NAMES}")


def collate_fn(batch):
    graphs, properties, labels = [], [], []
    for sample in batch:
        data = Data(
            x=sample["z"].float(),
            z=sample["z"],
            pos=sample["pos"],
            edge_index=sample["edge_index"],
            edge_attr=sample["edge_attr"],
        )
        graphs.append(data)

        vals = sample["properties"].numpy()
        if PROP_IMPUTER is not None:
            vals = PROP_IMPUTER.transform(vals.reshape(1, -1)).reshape(-1)
        if KEEP_COLS is not None:
            vals = vals[KEEP_COLS]
        if PROP_SCALER is not None:
            vals = PROP_SCALER.transform(vals.reshape(1, -1)).reshape(-1)

        properties.append(torch.tensor(vals, dtype=torch.float32))
        labels.append(sample["label"])

    batch_graphs = Batch.from_data_list(graphs)
    batch_properties = torch.stack(properties)
    batch_labels = torch.cat(labels)
    return batch_graphs, batch_properties, batch_labels


### A7. Optimizer / Scheduler, Class-Balanced Focal Loss

In [ ]:
# ================================================================
# ================================================================
def _build_optimizer_and_scheduler(model, lr=5e-4, total_epochs=300, wd=1e-4, warmup_epochs=10):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_norm = any(k in n for k in ["bias", "LayerNorm.weight", "LayerNorm.bias", "BatchNorm"])
        (no_decay if is_norm else decay).append(p)

    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}],
        lr=lr,
    )

    warmup_epochs = max(5, int(0.1 * total_epochs))

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler


def _build_optimizer_and_scheduler_film(model, base_lr=5e-4, total_epochs=300,
                                        warmup_epochs=10, prop_lr_mult=2.0,
                                        prop_boost_epochs=30, wd=1e-4):
    prop_params, other_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if ("prop." in n) or ("to_gamma" in n) or ("to_beta" in n):
            prop_params.append(p)
        else:
            other_params.append(p)

    optimizer = torch.optim.AdamW([
        {"params": other_params, "lr": base_lr, "weight_decay": wd},
        {"params": prop_params,  "lr": base_lr * prop_lr_mult, "weight_decay": wd * 0.5},
    ])

    def lr_lambda_other(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    def lr_lambda_prop(epoch):
        w = lr_lambda_other(epoch)
        return w * (prop_lr_mult if epoch < prop_boost_epochs else 1.0)

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=[lr_lambda_other, lr_lambda_prop]
    )
    return optimizer, scheduler


# ================================================================
# Class-balanced Focal Loss (Cui et al. CB Loss + Focal term)
# ================================================================
class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.999, gamma=1.5, num_classes=2):
        super().__init__()
        counts = np.array(class_counts, dtype=np.float64)
        counts = np.maximum(counts, 1.0)
        effective_num = 1.0 - np.power(beta, counts)
        weights = (1.0 - beta) / effective_num
        weights = weights / weights.sum() * num_classes
        self.register_buffer("weights", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, logits, targets):
        log_probs = torch.nn.functional.log_softmax(logits, dim=1)
        t = torch.nn.functional.one_hot(targets, num_classes=logits.size(1)).float()
        log_pt = (t * log_probs).sum(dim=1)
        pt = log_pt.exp().clamp(1e-6, 1 - 1e-6)
        focal = (1 - pt).pow(self.gamma) * (-log_pt)
        w = self.weights[targets]
        return (w * focal).mean()


def _get_class_counts_from_loader(train_loader, num_classes):
    counts = np.zeros(num_classes, dtype=int)
    for _, _, labels in train_loader:
        l = labels.numpy()
        binc = np.bincount(l, minlength=num_classes)
        counts[:len(binc)] += binc
    return counts.tolist()


In [ ]:
# ================================================================
# ================================================================
USE_TTA = True
TTA_K = 12
TTA_SIGMA = 0.015
TTA_MC_DROPOUT = True

USE_TTA_FOR_VAL = True
VAL_TTA_K = 8
VAL_TTA_SIGMA = 0.01
VAL_TTA_MC = True


@torch.no_grad()
def evaluate_model_tta(model, loader, K=12, coord_sigma=0.015, mc_dropout=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    if mc_dropout:
        for m in model.modules():
            if isinstance(m, nn.Dropout):
                m.train()

    all_predictions, all_labels, all_probs = [], [], []

    for batch_graphs, batch_properties, batch_labels in loader:
        batch_properties = batch_properties.to(device)
        logits_sum = None

        for _ in range(K):
            glist = batch_graphs.to_data_list()
            new_list = []
            for d in glist:
                pos = d.pos
                pos2 = pos + coord_sigma * torch.randn_like(pos) if coord_sigma and coord_sigma > 0 else pos.clone()
                row, col = d.edge_index
                ea = (pos2[row] - pos2[col]).pow(2).sum(dim=1).sqrt()
                d2 = Data(
                    x=getattr(d, "x", d.z.float()),
                    z=d.z,
                    pos=pos2,
                    edge_index=d.edge_index,
                    edge_attr=ea,
                )
                new_list.append(d2)

            gb = Batch.from_data_list(new_list).to(device)
            out = model(gb, batch_properties)
            logits_sum = out if logits_sum is None else (logits_sum + out)

        logits = logits_sum / float(K)
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)

        all_predictions.extend(pred.cpu().numpy())
        all_labels.extend(batch_labels.numpy())
        all_probs.extend(probs.cpu().numpy())

    return {
        "accuracy": accuracy_score(all_labels, all_predictions),
        "precision": precision_score(all_labels, all_predictions, average="weighted", zero_division=0),
        "recall": recall_score(all_labels, all_predictions, average="weighted", zero_division=0),
        "f1_score": f1_score(all_labels, all_predictions, average="weighted", zero_division=0),
        "predictions": all_predictions,
        "true_labels": all_labels,
        "probabilities": all_probs,
    }


def evaluate_unified(model, loader):
    if USE_TTA:
        return evaluate_model_tta(model, loader, K=TTA_K, coord_sigma=TTA_SIGMA, mc_dropout=TTA_MC_DROPOUT)
    return evaluate_model_tta(model, loader, K=1, coord_sigma=0.0, mc_dropout=False)


def train_model(model, train_loader, val_loader, model_name,
                num_epochs=300, lr=5e-4, save_dir="."):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class_counts = _get_class_counts_from_loader(train_loader, num_classes=NUM_CLASSES)
    criterion = ClassBalancedFocalLoss(
        class_counts, beta=0.999, gamma=1.5, num_classes=NUM_CLASSES
    ).to(device)

    if isinstance(model, SchNetTasteClassifier_FiLM) and model.use_properties:
        optimizer, scheduler = _build_optimizer_and_scheduler_film(
            model, base_lr=lr, total_epochs=num_epochs,
            warmup_epochs=10, prop_lr_mult=2.0,
            prop_boost_epochs=30, wd=1e-4
        )
    else:
        optimizer, scheduler = _build_optimizer_and_scheduler(
            model, lr=lr, total_epochs=num_epochs, wd=1e-4, warmup_epochs=10
        )

    best_val_macro_f1 = -float("inf")
    early_stopping_counter = 0
    early_stopping_patience = 20
    delta = 1e-4

    train_losses, val_losses, val_accuracies = [], [], []

    print(f"\n=== Training {model_name} ===\nTotal epochs: {num_epochs}")

    tag = model_name.lower().replace(" ", "_").replace("+", "plus").replace("(", "").replace(")", "")
    save_path = os.path.join(save_dir, f"{tag}_model.pth")

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        correct_train = 0
        total_train = 0

        for batch_graphs, batch_properties, batch_labels in train_loader:
            batch_graphs = batch_graphs.to(device)
            batch_properties = batch_properties.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            outputs = model(batch_graphs, batch_properties)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_train_loss += loss.item()
            pred = outputs.argmax(1)
            correct_train += (pred == batch_labels).sum().item()
            total_train += batch_labels.size(0)

        train_acc = 100 * correct_train / max(1, total_train)
        avg_train_loss = total_train_loss / max(1, len(train_loader))
        train_losses.append(avg_train_loss)

        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for batch_graphs, batch_properties, batch_labels in val_loader:
                batch_graphs = batch_graphs.to(device)
                batch_properties = batch_properties.to(device)
                out = model(batch_graphs, batch_properties)
                loss = criterion(out, batch_labels.to(device))
                total_val_loss += loss.item()

        if USE_TTA_FOR_VAL:
            val_eval = evaluate_model_tta(model, val_loader, K=VAL_TTA_K, coord_sigma=VAL_TTA_SIGMA, mc_dropout=VAL_TTA_MC)
        else:
            val_eval = evaluate_model_tta(model, val_loader, K=1, coord_sigma=0.0, mc_dropout=False)
        val_acc = 100 * val_eval["accuracy"]
        val_f1_macro = f1_score(val_eval["true_labels"], val_eval["predictions"], average="macro", zero_division=0)

        avg_val_loss = total_val_loss / max(1, len(val_loader))
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_acc)

        scheduler.step()

        if val_f1_macro > best_val_macro_f1 + delta:
            best_val_macro_f1 = val_f1_macro
            early_stopping_counter = 0
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(model.state_dict(), save_path)
        else:
            early_stopping_counter += 1

        if epoch % 10 == 0 or epoch < 5:
            try:
                lr_now = scheduler.get_last_lr()[0]
            except Exception:
                lr_now = optimizer.param_groups[0]["lr"]
            print(
                f"Epoch [{epoch:3d}/{num_epochs}] | "
                f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
                f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}% | "
                f"Val MacroF1: {val_f1_macro:.4f} | Best MacroF1: {best_val_macro_f1:.4f} | "
                f"LR: {lr_now:.2e}"
            )

        if early_stopping_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch} (no improvement >{delta} for {early_stopping_patience} epochs)")
            break

    print(f"{model_name} - Final Best Validation Macro-F1: {best_val_macro_f1:.4f}")
    return train_losses, val_accuracies, val_losses, best_val_macro_f1, save_path


In [ ]:
# ================================================================
# ================================================================
@torch.no_grad()
def permutation_importance_props(model, test_loader, base_metrics,
                                 n_repeats=10, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    data_list, props_list, labels_list = [], [], []
    for g, p, y in test_loader:
        data_list.extend(g.to_data_list())
        props_list.append(p)
        labels_list.append(y)

    props_full  = torch.cat(props_list, dim=0).to(device)
    labels_full = torch.cat(labels_list, dim=0).to(device)
    N = len(data_list)
    assert N == props_full.size(0) == labels_full.size(0)

    def eval_with_props(props_tensor):
        logits_chunks = []
        bs = 256
        for start in range(0, N, bs):
            end = min(start + bs, N)
            sub_batch = Batch.from_data_list(data_list[start:end]).to(device)
            sub_props = props_tensor[start:end]
            logits_chunks.append(model(sub_batch, sub_props))
        logits = torch.cat(logits_chunks, dim=0)
        pred = logits.argmax(dim=1)
        acc = (pred == labels_full).float().mean().item()
        f1m = f1_score(
            labels_full.detach().cpu().numpy(),
            pred.detach().cpu().numpy(),
            average='macro', zero_division=0
        )
        return acc, f1m

    base_acc, base_f1 = base_metrics['accuracy'], base_metrics['macro_f1']

    rng = np.random.default_rng(0)
    P = props_full.shape[1]
    rows = []
    for j in range(P):
        drops = []
        for _ in range(n_repeats):
            perm = props_full.clone()
            idx = torch.from_numpy(rng.permutation(N)).to(device)
            perm[:, j] = perm[:, j][idx]
            acc, f1m = eval_with_props(perm)
            drops.append(base_f1 - f1m)
        rows.append((j, float(np.mean(drops)), float(np.std(drops))))
    df = pd.DataFrame(rows, columns=["prop_idx", "macroF1_drop_mean", "macroF1_drop_std"])
    return df


def gradient_sensitivity_props(model, test_loader, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    grads_sum = None
    n_total = 0
    for g, p, y in test_loader:
        g = g.to(device)
        p = p.to(device).clone().detach().requires_grad_(True)
        logits = model(g, p)
        probs = torch.softmax(logits, dim=1)
        chosen = probs.max(dim=1).values.sum()
        chosen.backward()
        grad = p.grad.detach().abs()
        if grads_sum is None:
            grads_sum = grad.sum(dim=0)
        else:
            grads_sum += grad.sum(dim=0)
        n_total += grad.shape[0]
    sens = (grads_sum / max(1, n_total)).cpu().numpy()
    df = pd.DataFrame({"prop_idx": list(range(len(sens))), "avg_abs_grad": sens})
    return df


def _mask_coords_by_smarts(sample, mol, smarts, shift=50.0):
    patt = Chem.MolFromSmarts(smarts)
    if patt is None:
        return None
    matches = mol.GetSubstructMatches(patt)
    if not matches:
        return None
    pos = sample['pos'].clone().numpy()
    for match in matches:
        for a in match:
            pos[a, 0] += shift
            pos[a, 1] += shift
            pos[a, 2] += shift
    return _build_sample_from_mol_and_pos(
        mol, pos, label=int(sample['label'].item()), edge_cutoff=10.0
    )


@torch.no_grad()
def smarts_ablation_effect(model, dataset, test_indices,
                           smarts_list=None, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    if smarts_list is None:
        smarts_list = [
            "[c]1ccccc1",                        # benzene
            "[#6][OX2H]",                        # alcohol
            "[$([OH]-*=[!#6])]",                 # hydroxyl acidic
            "[NX3;H2,H1;!$(NC=O)]",              # primary/secondary amine
            "C(=O)O",                            # carboxylate-type
            "[NX3][CX3](=[OX1])[#6]",            # amide
            "[S](=O)(=O)N",                      # sulfonamide
            "[F,Cl,Br,I]",                       # halogen
            "[OD2]([#6])[#6]",                   # ether
            "[$([NX3H2,NX4H3+]),$([NX3H](C)(C))][CX4H]([*])[CX3](=[OX1])[OX2H,OX1-,N]"
        ]
    rows = []
    for smarts in smarts_list:
        deltas = []
        n_match = 0
        for idx in test_indices:
            s = dataset.data[idx]
            if 'mol_idx' not in s:
                continue
            mol = dataset.mols[s['mol_idx']]
            masked = _mask_coords_by_smarts(s, mol, smarts, shift=40.0)
            if masked is None:
                continue
            n_match += 1
            g0, p0 = collate_fn([s])[:2]
            g1, p1 = collate_fn([masked])[:2]
            g0 = g0.to(device); p0 = p0.to(device)
            g1 = g1.to(device); p1 = p1.to(device)
            out0 = torch.softmax(model(g0, p0), dim=1)[0]
            out1 = torch.softmax(model(g1, p1), dim=1)[0]
            gt = int(s['label'].item())
            delta = float(out0[gt] - out1[gt])
            deltas.append(delta)
        if n_match > 0:
            rows.append([smarts, n_match, float(np.mean(deltas)), float(np.std(deltas))])
    df = pd.DataFrame(rows, columns=["SMARTS", "n_matched", "mean_delta_prob_GT", "std_delta"])
    return df


In [ ]:
# ================================================================
# ================================================================
def plot_comparison_results(results_dict, training_histories, out_dir="plots", tag=""):
    os.makedirs(out_dir, exist_ok=True)
    models = list(results_dict.keys())
    metrics = ["accuracy", "precision", "recall", "f1_score"]
    colors = ["#f3993b", "#b97af7", "#5fbcd3", "#d35f5f", "#7f7f7f", "#8dd35f"]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for i, metric in enumerate(metrics):
        row, col = i // 2, i % 2
        values = [results_dict[m][metric] for m in models]
        bars = axes[row, col].bar(models, values, color=colors[:len(models)], alpha=0.8)
        axes[row, col].set_title(f"{metric.title()} Comparison ({tag})", fontsize=12, fontweight="bold")
        axes[row, col].set_ylabel(metric.title())
        axes[row, col].set_ylim(0, max(values) * 1.10 if len(values) else 1.0)
        for bar, value in zip(bars, values):
            axes[row, col].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontweight="bold",
            )

    axes[0, 2].set_title(f"Training Loss ({tag})")
    axes[1, 2].set_title(f"Validation Accuracy ({tag})")
    axes[0, 2].set_xlabel("Epoch")
    axes[1, 2].set_xlabel("Epoch")
    axes[0, 2].set_ylabel("Loss")
    axes[1, 2].set_ylabel("Accuracy (%)")

    keys2 = list(training_histories.keys())[:2]
    for i, model_name in enumerate(keys2):
        epochs = range(len(training_histories[model_name]["train_losses"]))
        axes[0, 2].plot(epochs, training_histories[model_name]["train_losses"], label=model_name, color=colors[i])
        axes[1, 2].plot(epochs, training_histories[model_name]["val_accuracies"], label=model_name, color=colors[i])

    axes[0, 2].legend()
    axes[1, 2].legend()
    axes[0, 2].grid(True)
    axes[1, 2].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"schnet_metrics_comparison_{tag}.pdf"), dpi=500)
    plt.show()

    df_results = pd.DataFrame(results_dict).T
    print("\n" + "=" * 60)
    print(f"SchNet Model Performance Comparison ({tag})")
    print("=" * 60)
    print(df_results.round(4))
    return df_results


def build_models_dict():
    return {
        "SchNet (3D Only)": SchNetTasteClassifier_FiLM(
            schnet_hidden=128, property_dim=PROPERTY_DIM,
            num_classes=NUM_CLASSES, use_properties=False
        ),
        "SchNet (3D + Properties)": SchNetTasteClassifier_FiLM(
            schnet_hidden=128, property_dim=PROPERTY_DIM,
            num_classes=NUM_CLASSES, use_properties=True
        ),
    }


def save_confusion_matrix(y_true, y_pred, class_names=None, out_png="cm.png", title="Confusion Matrix"):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    C = len(sorted(set(y_true) | set(y_pred)))

    if class_names is None:
        if C == 2:
            class_names = ["Bitter", "Sweet"]
        elif C == 3:
            class_names = ["Bitter", "Sweet", "Tasteless"]
        else:
            class_names = [str(i) for i in range(C)]

    labels = list(range(C))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(C),
        yticks=np.arange(C),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True",
        xlabel="Predicted",
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = cm.max() * 0.5 if cm.size > 0 else 0
    for i in range(C):
        for j in range(C):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    fig.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close(fig)


# ================================================================
# ================================================================
def check_split(labels, groups, train_idx, val_idx, test_idx, name=""):
    def stat(idx):
        y = labels[idx]
        return np.bincount(y, minlength=NUM_CLASSES)

    print(f"\n[CHECK {name}] sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
    print("[CHECK] class dist - train/val/test:", stat(train_idx), stat(val_idx), stat(test_idx))

    g = np.array(groups)
    gtr, gval, gte = set(g[train_idx]), set(g[val_idx]), set(g[test_idx])
    inter_tr_val = gtr & gval
    inter_tr_te = gtr & gte
    inter_val_te = gval & gte
    print(f"[LEAK] overlap train-val={len(inter_tr_val)}, train-test={len(inter_tr_te)}, val-test={len(inter_val_te)}")
    assert len(inter_tr_val) == 0 and len(inter_tr_te) == 0 and len(inter_val_te) == 0, "Group leakage detected!"


def make_best_sgkf_split(labels, groups, n_splits=5, n_trials=80, seed=42):
    labels = np.asarray(labels)
    groups = np.asarray(groups)
    best = None
    rng = np.random.default_rng(seed)

    for _ in range(n_trials):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=int(rng.integers(1, 10**9)))
        fold_id = np.empty(len(labels), dtype=int)
        for f, (_, idx) in enumerate(sgkf.split(np.zeros(len(labels)), labels, groups)):
            fold_id[idx] = f

        ratios = []
        for f in range(n_splits):
            y_f = labels[fold_id == f]
            cnt = np.bincount(y_f, minlength=int(labels.max()) + 1)
            ratio = cnt / max(1, cnt.sum())
            ratios.append(ratio)

        ratios = np.stack(ratios, axis=0)
        score = ratios.std(axis=0).mean()

        if (best is None) or (score < best[0]):
            best = (score, fold_id)

    return best[1]


In [ ]:
# ================================================================
# ================================================================
def run_experiment(
    run_label,
    base_indices,
    do_augment=False,
    aug_mode="tasteless_only",
    tasteless_gap_ratio=1.0,
    num_epochs=300,
    lr=5e-4
):
    print("\n" + "#" * 80)
    print(f"Running experiment: {run_label} (augment={do_augment}, aug_mode={aug_mode})")
    print("#" * 80)

    ds = TasteDataset(
        SDF_PATH,
        label_map=LABEL_MAP,
        edge_cutoff=10.0,
    )

    train_idx = np.array(base_indices["train"])
    val_idx = np.array(base_indices["val"])
    test_idx = np.array(base_indices["test"])

    train_idx_aug = train_idx.copy()

    save_dir = os.path.join(SAVE_DIR_BASE, run_label)
    plots_dir = os.path.join(save_dir, "plots")
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    if do_augment:
        aug_sdf_path = os.path.join(save_dir, f"aug_{aug_mode}.sdf")
        aug_csv_path = os.path.join(save_dir, f"aug_{aug_mode}.csv")

        if aug_mode != "tasteless_only":
            raise ValueError("This notebook supports only the tasteless_only augmentation described in the paper.")
        if "tastelessness" not in LABEL_MAP:
            raise ValueError("AUG_MODE='tasteless_only' requires tastelessness in the current LABEL_MAP (ternary task only).")

        added_idx = augment_tasteless_only(
            dataset=ds,
            train_idx=train_idx,
            tasteless_label=LABEL_MAP["tastelessness"],
            max_extra_per_mol=AUG_MAX_EXTRA_PER_MOL,
            gap_ratio=tasteless_gap_ratio,
            rmsd_thresh=AUG_RMSD_THRESH,
            noise_sigma=AUG_NOISE_SIGMA,
            seed=AUG_SEED,
            save_aug_sdf_path=aug_sdf_path,
            save_aug_csv_path=aug_csv_path,
        )

        if len(added_idx) > 0:
            train_idx_aug = np.concatenate([train_idx, np.array(added_idx, dtype=int)])

    _fit_property_preprocessors_safe(ds, train_idx_aug)

    batch_size = 32
    if do_augment:
        train_loader = DataLoader(
            Subset(ds, train_idx_aug),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
        )
    else:
        train_labels = np.array([ds.data[i]["label"].item() for i in train_idx_aug])
        class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
        weights_per_class = 1.0 / np.maximum(class_counts, 1)
        sample_weights = torch.DoubleTensor([weights_per_class[y] for y in train_labels])
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(
            Subset(ds, train_idx_aug),
            batch_size=batch_size,
            sampler=sampler,
            collate_fn=collate_fn,
        )

    val_loader = DataLoader(Subset(ds, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(Subset(ds, test_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    print(f"[{run_label}] Sizes - Train: {len(train_idx_aug)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

    models = build_models_dict()
    all_results = {}
    training_histories = {}

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for model_name, model in models.items():
        print("\n" + "=" * 60)
        print(f"Training {model_name} ({run_label})")
        print("=" * 60)
        print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
        if "Properties" in model_name:
            print(f"Input: 3D + {PROPERTY_DIM} props (KEEP_COLS={FINAL_PROP_NAMES})")
        else:
            print("Input: 3D only")

        train_losses, val_accuracies, val_losses, best_val_macro_f1, model_path = train_model(
            model, train_loader, val_loader,
            model_name, num_epochs=num_epochs, lr=lr, save_dir=save_dir
        )

        state = torch.load(model_path, map_location=device)
        model.load_state_dict(state)

        results = evaluate_unified(model, test_loader)

        key = f"{model_name} [{run_label}]"
        all_results[key] = results
        training_histories[key] = {
            "train_losses": train_losses,
            "val_accuracies": val_accuracies,
            "val_losses": val_losses,
            "best_val_macro_f1": best_val_macro_f1,
        }

        print(f"\n{model_name} [{run_label}] Test Results:")
        print(f"   Accuracy:  {results['accuracy']:.4f}")
        print(f"   Precision: {results['precision']:.4f}")
        print(f"   Recall:    {results['recall']:.4f}")
        print(f"   F1-Score:  {results['f1_score']:.4f}")

    _ = plot_comparison_results(all_results, training_histories, out_dir=plots_dir, tag=run_label)

    json_path = os.path.join(save_dir, f"results_{run_label}.json")
    with open(json_path, "w") as f:
        json.dump({
            "results": {
                k: {m: float(v) for m, v in res.items()
                    if m not in ["predictions", "true_labels", "probabilities"]}
                for k, res in all_results.items()
            },
            "training_histories": training_histories,
            "config": {
                "RUN_BINARY": RUN_BINARY,
                "NUM_CLASSES": NUM_CLASSES,
                "LABEL_MAP": LABEL_MAP,
                "DO_AUGMENT": do_augment,
                "AUG_MODE": aug_mode,
                "USE_TTA": USE_TTA,
                "TTA_K": TTA_K,
                "TTA_SIGMA": TTA_SIGMA,
                "TTA_MC_DROPOUT": TTA_MC_DROPOUT,
            }
        }, f, indent=2)
    print(f"Saved: {json_path}")

    return ds, all_results, training_histories


def prepare_split_and_run(run_label, num_epochs=300, lr=5e-4):
    base_ds = TasteDataset(SDF_PATH, label_map=LABEL_MAP, edge_cutoff=10.0)
    labels = np.array([s["label"].item() for s in base_ds.data])
    groups = np.array([s["group"] for s in base_ds.data])

    fold_id = make_best_sgkf_split(labels, groups, n_splits=5, n_trials=80, seed=42)
    train_folds, val_fold, test_fold = [0, 1, 2], 3, 4

    train_idx = np.where(np.isin(fold_id, train_folds))[0]
    val_idx = np.where(fold_id == val_fold)[0]
    test_idx = np.where(fold_id == test_fold)[0]

    base_indices = {"train": train_idx.tolist(), "val": val_idx.tolist(), "test": test_idx.tolist()}
    os.makedirs(SAVE_DIR_BASE, exist_ok=True)
    with open(os.path.join(SAVE_DIR_BASE, f"indices_{run_label}.json"), "w") as f:
        json.dump(base_indices, f, indent=2)

    check_split(labels, groups, train_idx, val_idx, test_idx, name=f"final split ({run_label})")

    ds, results, histories = run_experiment(
        run_label=run_label,
        base_indices=base_indices,
        do_augment=DO_AUGMENT,
        aug_mode=AUG_MODE,
        tasteless_gap_ratio=AUG_GAP_RATIO,
        num_epochs=num_epochs,
        lr=lr,
    )

    key_prop = f"SchNet (3D + Properties) [{run_label}]"
    cm_png = os.path.join(SAVE_DIR_BASE, run_label, f"confusion_matrix_test_{run_label}.png")

    class_names = ["Bitter", "Sweet"] if NUM_CLASSES == 2 else ["Bitter", "Sweet", "Tasteless"]

    save_confusion_matrix(
        results[key_prop]["true_labels"],
        results[key_prop]["predictions"],
        class_names=class_names,
        out_png=cm_png,
        title=f"Confusion Matrix - {key_prop} (TTA={USE_TTA})",
    )
    print("Saved confusion matrix:", cm_png)

    return ds, results, histories


In [ ]:
# ------------------------------------------------------------
# ------------------------------------------------------------
set_mode_binary()
# set_mode_triclass()

# AUG_MODE = "tasteless_only"

aug_suffix = f"aug_{AUG_MODE}" if DO_AUGMENT else "noaug"
run_label = f"{'binary' if RUN_BINARY else 'triclass'}_{aug_suffix}"

print("RUN START:", run_label)
print("LABEL_MAP:", LABEL_MAP, "| NUM_CLASSES:", NUM_CLASSES, "| DO_AUGMENT:", DO_AUGMENT)

# ds, results, histories = prepare_split_and_run(run_label=run_label, num_epochs=300, lr=5e-4)


In [ ]:
# ================================================================
# ================================================================
TASK = "binary"

FOLD_JSON_PATH = "analysis/taste_baseline_cv/binary/folds_binary_seed42.json"
OUTPUT_DIR = "analysis/taste_baseline_cv/binary/schnet_seed42"

NUM_EPOCHS = 300
LEARNING_RATE = 5e-4
BATCH_SIZE = 32
INNER_VAL_FRACTION = 0.20
BOOTSTRAP_REPEATS = 5000

USE_TTA_FOR_VAL = False
USE_TTA = False
TTA_MC_DROPOUT = False
VAL_TTA_MC = False


In [ ]:
# ================================================================
# ================================================================
class TasteDatasetAligned(Dataset):
    def __init__(self, sdf_file, transform=None,
                 label_map=None, edge_cutoff=10.0,
                 try_embed_if_no_conf=True, random_seed=0xF00D):
        self.transform = transform
        self.data = []
        self.edge_cutoff = edge_cutoff
        self.mols = []

        if label_map is None:
            label_map = LABEL_MAP

        norm = {
            'bitter': 'bitterness', 'bitterness': 'bitterness',
            'sweet': 'sweetness', 'sweetness': 'sweetness',
            'tasteless': 'tastelessness', 'tastelessness': 'tastelessness',
            'no taste': 'tastelessness', 'notaste': 'tastelessness',
            'none': 'tastelessness', 'tastelessness ': 'tastelessness',
        }
        self.label_map = {k.lower().strip(): int(v) for k, v in label_map.items()}

        suppl = Chem.SDMolSupplier(sdf_file, sanitize=False, removeHs=False)

        cnt_total = cnt_kept = 0
        cnt_none = cnt_no_label = cnt_not_in_map = cnt_no_conf = 0
        seen_raw_labels = {}

        def normalize_label(raw):
            key = raw.strip().lower()
            key = norm.get(key, key)
            return key

        def get_class_taste_prop(m):
            if m.HasProp('Class taste'):
                return 'Class taste'
            for p in m.GetPropNames():
                if p.strip().lower() == 'class taste':
                    return p
            return None

        for mol in suppl:
            cnt_total += 1
            if mol is None:
                cnt_none += 1
                continue

            try:
                Chem.SanitizeMol(mol)
            except Exception:
                try:
                    Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_KEKULIZE)
                except Exception:
                    cnt_none += 1
                    continue

            taste_prop = get_class_taste_prop(mol)
            if taste_prop is None:
                cnt_no_label += 1
                continue

            raw = mol.GetProp(taste_prop)
            seen_raw_labels[raw] = seen_raw_labels.get(raw, 0) + 1

            taste_key = normalize_label(raw)
            if taste_key not in self.label_map:
                cnt_not_in_map += 1
                continue
            label = self.label_map[taste_key]

            try:
                base_mol = Chem.RemoveHs(mol)
                _ = Chem.MolToSmiles(base_mol, canonical=True)
                _ = AllChem.GetMorganFingerprintAsBitVect(base_mol, 2, nBits=2048)
            except Exception:
                cnt_none += 1
                continue

            if mol.GetNumConformers() == 0:
                if not try_embed_if_no_conf:
                    cnt_no_conf += 1
                    continue
                mH = Chem.AddHs(mol, addCoords=True)
                params = AllChem.ETKDGv3()
                params.randomSeed = random_seed
                if AllChem.EmbedMolecule(mH, params) != 0:
                    cnt_no_conf += 1
                    continue
                try:
                    AllChem.UFFOptimizeMolecule(mH)
                except Exception:
                    pass
                mol = mH

            conf = mol.GetConformer()
            pos = conf.GetPositions()
            if pos is None or len(pos) == 0:
                cnt_no_conf += 1
                continue

            atomic_nums = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
            n = len(pos)
            edge_indices, edge_attrs = [], []
            for a in range(n):
                for b in range(a + 1, n):
                    dist = float(np.linalg.norm(pos[a] - pos[b]))
                    if dist < self.edge_cutoff:
                        edge_indices.extend([[a, b], [b, a]])
                        edge_attrs.extend([dist, dist])
            if len(edge_indices) == 0:
                for k in range(n):
                    edge_indices.append([k, k])
                    edge_attrs.append(0.0)

            vals, miss = [], []
            for pn in BASE_PROP_NAMES:
                try:
                    v = float(mol.GetProp(pn))
                    if np.isnan(v):
                        vals.append(np.nan); miss.append(1.0)
                    else:
                        vals.append(v);      miss.append(0.0)
                except Exception:
                    vals.append(np.nan);     miss.append(1.0)

            can_smiles = Chem.MolToSmiles(Chem.RemoveHs(mol))
            self.mols.append(Chem.Mol(mol))
            mol_idx = len(self.mols) - 1

            sample = {
                'pos': torch.tensor(pos, dtype=torch.float32),
                'z': torch.tensor(atomic_nums, dtype=torch.long),
                'edge_index': torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                'edge_attr': torch.tensor(edge_attrs, dtype=torch.float32),
                'properties': torch.tensor(vals, dtype=torch.float32),
                'prop_missing': torch.tensor(miss, dtype=torch.float32),
                'label': torch.tensor([label], dtype=torch.long),
                'group': can_smiles,
                'mol_idx': mol_idx,
            }
            self.data.append(sample)
            cnt_kept += 1

        print(f"[TasteDatasetAligned] total={cnt_total}, kept={cnt_kept}, "
              f"none={cnt_none}, no_label={cnt_no_label}, not_in_map={cnt_not_in_map}, no_conf={cnt_no_conf}")
        if seen_raw_labels:
            print("[TasteDatasetAligned] Raw label histogram:",
                  {k: v for k, v in sorted(seen_raw_labels.items(), key=lambda x: -x[1])})

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        if self.transform:
            sample = self.transform(sample)
        return sample


In [ ]:
from pathlib import Path
from scipy.stats import binomtest


def set_reproducible_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def configure_task_oof(task: str):
    global RUN_BINARY, LABEL_MAP, NUM_CLASSES
    task = task.lower().strip()
    if task == "binary":
        RUN_BINARY = True
        LABEL_MAP = LABEL_MAP_BIN
    elif task == "triclass":
        RUN_BINARY = False
        LABEL_MAP = LABEL_MAP_TRI
    else:
        raise ValueError("TASK must be 'binary' or 'triclass'.")
    NUM_CLASSES = len(LABEL_MAP)
    return LABEL_MAP


def load_outer_folds(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if "folds" not in obj or len(obj["folds"]) == 0:
        raise ValueError("fold JSON must contain a non-empty 'folds' list.")
    return obj


def split_inner_train_val(ds, outer_train_idx, seed, val_fraction=0.20):
    outer_train_idx = np.asarray(outer_train_idx, dtype=int)
    labels = np.asarray([ds.data[i]["label"].item() for i in outer_train_idx])
    groups = np.asarray([ds.data[i]["group"] for i in outer_train_idx])

    n_splits = max(2, int(round(1.0 / val_fraction)))
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )
    inner_train_local, val_local = next(
        splitter.split(np.zeros(len(outer_train_idx)), labels, groups)
    )
    return outer_train_idx[inner_train_local], outer_train_idx[val_local]


def make_fold_loaders(ds, train_idx, val_idx, test_idx, batch_size=32):
    train_idx = np.asarray(train_idx, dtype=int)
    val_idx = np.asarray(val_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    _fit_property_preprocessors_safe(ds, train_idx)

    y_train = np.asarray([ds.data[i]["label"].item() for i in train_idx])
    counts = np.bincount(y_train, minlength=NUM_CLASSES)
    inverse = 1.0 / np.maximum(counts, 1)
    sample_weights = torch.DoubleTensor([inverse[y] for y in y_train])
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

    train_loader = DataLoader(
        Subset(ds, train_idx), batch_size=batch_size,
        sampler=sampler, collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        Subset(ds, val_idx), batch_size=batch_size,
        shuffle=False, collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        Subset(ds, test_idx), batch_size=batch_size,
        shuffle=False, collate_fn=collate_fn,
    )
    return train_loader, val_loader, test_loader


def metric_dict(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def paired_bootstrap_difference(y_true, pred_a, pred_b, metric="f1_macro", B=5000, seed=42):
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    rng = np.random.default_rng(seed)
    n = len(y_true)

    def score(y, p):
        if metric == "accuracy":
            return accuracy_score(y, p)
        if metric == "f1_weighted":
            return f1_score(y, p, average="weighted", zero_division=0)
        if metric == "f1_macro":
            return f1_score(y, p, average="macro", zero_division=0)
        raise ValueError(metric)

    observed = score(y_true, pred_a) - score(y_true, pred_b)
    diffs = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        diffs[b] = score(y_true[idx], pred_a[idx]) - score(y_true[idx], pred_b[idx])

    lo, hi = np.quantile(diffs, [0.025, 0.975])
    p_two_sided = min(1.0, 2.0 * min(np.mean(diffs <= 0), np.mean(diffs >= 0)))
    return {
        "metric": metric,
        "difference_3d_property_minus_3d_only": observed,
        "ci95_low": lo,
        "ci95_high": hi,
        "p_bootstrap_two_sided": p_two_sided,
        "significant_0.05": bool((lo > 0) or (hi < 0)),
    }


def exact_mcnemar(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true)
    correct_a = np.asarray(pred_a) == y_true
    correct_b = np.asarray(pred_b) == y_true
    n01 = int(np.sum((~correct_a) & correct_b))
    n10 = int(np.sum(correct_a & (~correct_b)))
    discordant = n01 + n10
    p = 1.0 if discordant == 0 else binomtest(min(n01, n10), discordant, 0.5, alternative="two-sided").pvalue
    return {"n01_a_wrong_b_right": n01, "n10_a_right_b_wrong": n10, "p_exact": p}


In [ ]:
def run_schnet_oof(
    task=TASK,
    sdf_path=SDF_PATH,
    fold_json_path=FOLD_JSON_PATH,
    output_dir=OUTPUT_DIR,
    base_seed=SEED,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    inner_val_fraction=INNER_VAL_FRACTION,
    bootstrap_repeats=BOOTSTRAP_REPEATS,
):
    configure_task_oof(task)
    set_reproducible_seed(base_seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    ds = TasteDatasetAligned(sdf_path, label_map=LABEL_MAP, edge_cutoff=10.0)
    fold_obj = load_outer_folds(fold_json_path)
    n = len(ds)

    all_test_indices = sorted(
        int(i) for fold_item in fold_obj["folds"] for i in fold_item["test"]
    )
    expected_indices = list(range(n))
    if all_test_indices != expected_indices:
        missing = sorted(set(expected_indices) - set(all_test_indices))
        extra = sorted(set(all_test_indices) - set(expected_indices))
        raise RuntimeError(
            "Fold JSON and SchNet dataset are not aligned. "
            f"dataset_n={n}, fold_test_unique={len(set(all_test_indices))}, "
            f"missing={missing[:10]}, extra={extra[:10]}"
        )
    print(f"[alignment check] dataset_n={n}; fold indices cover 0..{n-1} exactly")

    model_names = ["SchNet (3D Only)", "SchNet (3D + Properties)"]
    oof_true = np.full(n, -1, dtype=int)
    oof_fold = np.full(n, -1, dtype=int)
    oof_pred = {name: np.full(n, -1, dtype=int) for name in model_names}
    fold_metric_rows = []

    for fold_item in fold_obj["folds"]:
        fold = int(fold_item["fold"])
        outer_train = np.asarray(fold_item["train"], dtype=int)
        outer_test = np.asarray(fold_item["test"], dtype=int)
        inner_train, inner_val = split_inner_train_val(
            ds, outer_train, base_seed + fold, inner_val_fraction
        )

        g = np.asarray([d["group"] for d in ds.data])
        assert not (set(g[inner_train]) & set(g[inner_val]))
        assert not (set(g[inner_train]) & set(g[outer_test]))
        assert not (set(g[inner_val]) & set(g[outer_test]))

        y_test = np.asarray([ds.data[i]["label"].item() for i in outer_test])
        oof_true[outer_test] = y_test
        oof_fold[outer_test] = fold

        print("\n" + "=" * 88)
        print(f"TASK={task} | fold={fold} | train={len(inner_train)} val={len(inner_val)} test={len(outer_test)}")
        print("=" * 88)

        for model_order, model_name in enumerate(model_names):
            set_reproducible_seed(base_seed + fold * 100 + model_order)
            train_loader, val_loader, test_loader = make_fold_loaders(
                ds, inner_train, inner_val, outer_test, batch_size
            )

            model = build_models_dict()[model_name]
            model_dir = output_dir / f"fold_{fold}" / (
                model_name.lower().replace(" ", "_").replace("+", "plus").replace("(", "").replace(")", "")
            )
            model_dir.mkdir(parents=True, exist_ok=True)

            history = train_model(
                model, train_loader, val_loader, model_name,
                num_epochs=num_epochs, lr=lr, save_dir=str(model_dir)
            )
            tag = model_name.lower().replace(" ", "_").replace("+", "plus").replace("(", "").replace(")", "")
            checkpoint = model_dir / f"{tag}_model.pth"
            if not checkpoint.exists():
                raise FileNotFoundError(checkpoint)
            model.load_state_dict(torch.load(checkpoint, map_location="cpu"))
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = model.to(device)

            # Deterministic outer-test inference for fair model comparison
            result = evaluate_model_tta(model, test_loader, K=1, coord_sigma=0.0, mc_dropout=False)
            pred = np.asarray(result["predictions"], dtype=int)
            true_from_loader = np.asarray(result["true_labels"], dtype=int)
            if not np.array_equal(true_from_loader, y_test):
                raise RuntimeError(f"Fold {fold}: test label order mismatch")
            oof_pred[model_name][outer_test] = pred

            met = metric_dict(y_test, pred)
            fold_metric_rows.append({"fold": fold, "model": model_name, **met})
            print(f"{model_name}: macro-F1={met['f1_macro']:.4f}, accuracy={met['accuracy']:.4f}")

    if np.any(oof_true < 0) or np.any(oof_fold < 0):
        raise RuntimeError("Some samples did not receive OOF predictions.")
    for name, pred in oof_pred.items():
        if np.any(pred < 0):
            raise RuntimeError(f"Missing OOF predictions for {name}")

    fold_metrics = pd.DataFrame(fold_metric_rows)
    fold_metrics.to_csv(output_dir / f"schnet_fold_metrics_{task}_seed{base_seed}.csv", index=False)

    oof_table = pd.DataFrame({
        "sample_idx": np.arange(n),
        "fold": oof_fold,
        "true_label": oof_true,
    })
    summary_rows = []
    external_paths = {}
    for name in model_names:
        col = "pred_3d_only" if "Only" in name else "pred_3d_property"
        oof_table[col] = oof_pred[name]
        summary_rows.append({"model": name, **metric_dict(oof_true, oof_pred[name])})
        external = pd.DataFrame({
            "sample_idx": np.arange(n),
            "true_label": oof_true,
            "pred_label": oof_pred[name],
        })
        external_path = output_dir / f"oof_{col.replace('pred_', '')}_{task}_seed{base_seed}.csv"
        external.to_csv(external_path, index=False)
        external_paths[name] = str(external_path)

    oof_table.to_csv(output_dir / f"schnet_oof_predictions_{task}_seed{base_seed}.csv", index=False)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(output_dir / f"schnet_oof_metrics_{task}_seed{base_seed}.csv", index=False)

    paired_rows = [
        paired_bootstrap_difference(
            oof_true,
            oof_pred["SchNet (3D + Properties)"],
            oof_pred["SchNet (3D Only)"],
            metric=m,
            B=bootstrap_repeats,
            seed=base_seed,
        )
        for m in ["accuracy", "f1_weighted", "f1_macro"]
    ]
    paired = pd.DataFrame(paired_rows)
    paired.to_csv(output_dir / f"paired_bootstrap_3dproperty_vs_3donly_{task}_seed{base_seed}.csv", index=False)

    mc = exact_mcnemar(
        oof_true,
        oof_pred["SchNet (3D Only)"],
        oof_pred["SchNet (3D + Properties)"],
    )
    pd.DataFrame([mc]).to_csv(output_dir / f"mcnemar_3donly_vs_3dproperty_{task}_seed{base_seed}.csv", index=False)

    print("\nOOF summary")
    print(summary)
    print("\nPaired bootstrap: 3D+Property minus 3D-only")
    print(paired)
    print("\nExact McNemar")
    print(pd.DataFrame([mc]))
    print("\nExternal prediction files")
    for name, p in external_paths.items():
        print(name, "->", p)

    return {
        "summary": summary,
        "fold_metrics": fold_metrics,
        "oof": oof_table,
        "paired_bootstrap": paired,
        "mcnemar": pd.DataFrame([mc]),
        "external_paths": external_paths,
    }


In [ ]:
# results = run_schnet_oof()
